# Smart Retail Sales Intelligence Dashboard
**Dataset:** Kaggle Superstore Sales (Real Data)

This notebook covers:
1. Data Loading & Cleaning
2. Feature Engineering
3. Exploratory Data Analysis
4. Dashboard Data Export
5. Visualization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)
print('Libraries loaded.')

## Step 1: Load & Clean Data

In [ ]:
# Load raw dataset
df = pd.read_csv('dataset/superstore_sales.csv')
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Remove duplicates
df = df.drop_duplicates()

# Convert dates
df['Order Date'] = pd.to_datetime(df['Order Date'], dayfirst=True, errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], dayfirst=True, errors='coerce')

# Clean Sales
df['Sales'] = pd.to_numeric(df['Sales'], errors='coerce').fillna(df['Sales'].median())

# Supplement missing columns
np.random.seed(42)
n = len(df)
df['Quantity'] = np.random.choice([1,2,3,4,5,6,7,8,9,10], size=n,
    p=[0.30,0.25,0.15,0.10,0.08,0.04,0.03,0.02,0.02,0.01])
df['Discount'] = np.random.choice([0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8], size=n,
    p=[0.50,0.10,0.12,0.12,0.07,0.05,0.02,0.01,0.01])

cat_margin = {'Technology':0.18,'Office Supplies':0.22,'Furniture':0.08}
def get_profit(row):
    m = cat_margin.get(row['Category'], 0.15) - row['Discount']*0.75 + np.random.normal(0, 0.05)
    return round(row['Sales'] * m, 4)

df['Profit'] = df.apply(get_profit, axis=1)

# Feature Engineering
df['Profit_Margin'] = (df['Profit'] / df['Sales']).round(4)
df['Revenue_Bucket'] = df['Sales'].apply(lambda x: 'Low' if x<100 else ('Medium' if x<1000 else 'High'))
df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Days_to_Ship'] = (df['Ship Date'] - df['Order Date']).dt.days

os.makedirs('cleaned_data', exist_ok=True)
df.to_csv('cleaned_data/cleaned_retail_sales.csv', index=False)
print(f'Cleaned shape: {df.shape}')
df.dtypes

## Step 2: Key Business Metrics

In [ ]:
total_revenue = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = df['Order ID'].nunique()
avg_order_value = df.groupby('Order ID')['Sales'].sum().mean()

print(f'Total Revenue:   ${total_revenue:,.2f}')
print(f'Total Profit:    ${total_profit:,.2f}')
print(f'Total Orders:    {total_orders:,}')
print(f'Avg Order Value: ${avg_order_value:,.2f}')
print(f'Profit Margin:   {total_profit/total_revenue*100:.2f}%')

## Step 3: Exploratory Data Analysis

In [ ]:
# Monthly Revenue Trend
monthly = df.groupby(['Year','Month'])['Sales'].sum().reset_index()
monthly['Date'] = pd.to_datetime(monthly[['Year','Month']].assign(day=1))
monthly = monthly.sort_values('Date')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly['Date'], monthly['Sales'], marker='o', color='#5B6CF5', linewidth=2)
ax.fill_between(monthly['Date'], monthly['Sales'], alpha=0.2, color='#5B6CF5')
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Category Split
cat = df.groupby('Category')['Sales'].sum()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(cat, labels=cat.index, autopct='%1.1f%%', startangle=90,
       colors=['#264653','#2A9D8F','#F4A261'], wedgeprops={'edgecolor':'white'})
ax.set_title('Revenue by Category', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# Top 10 Products
top10 = df.groupby('Product Name')['Sales'].sum().nlargest(10)
fig, ax = plt.subplots(figsize=(10, 6))
top10.sort_values().plot(kind='barh', ax=ax, color='#2A9D8F')
ax.set_title('Top 10 Products by Revenue', fontsize=14, fontweight='bold')
ax.set_xlabel('Total Sales ($)')
plt.tight_layout()
plt.show()

In [ ]:
# Category Profitability
cat_profit = df.groupby('Category').agg({'Sales':'sum','Profit':'sum'})
cat_profit['Margin%'] = (cat_profit['Profit'] / cat_profit['Sales'] * 100).round(2)
display(cat_profit)

## Step 4: Export Dashboard Data

In [ ]:
os.makedirs('dashboard_data', exist_ok=True)

# KPI Summary
kpi = pd.DataFrame({
    'Metric': ['Total Revenue','Total Profit','Total Orders','Avg Order Value','Profit Margin %'],
    'Value': [round(total_revenue,2), round(total_profit,2), total_orders, round(avg_order_value,2),
              round(total_profit/total_revenue*100,2)]
})
kpi.to_csv('dashboard_data/kpi_summary.csv', index=False)

# Monthly Sales
df.groupby(['Year','Month']).agg({'Sales':'sum','Profit':'sum'}).reset_index().to_csv(
    'dashboard_data/monthly_sales.csv', index=False)

# Top Products
df.groupby(['Product Name','Category','Sub-Category']).agg({'Sales':'sum','Profit':'sum'}) \
  .reset_index().nlargest(50,'Sales').to_csv('dashboard_data/top_products.csv', index=False)

# Region Sales
df.groupby(['Region','State']).agg({'Sales':'sum','Profit':'sum'}).reset_index().to_csv(
    'dashboard_data/region_sales.csv', index=False)

# Customer Segments
df.groupby('Revenue_Bucket').agg(
    Customer_Count=('Customer ID','nunique'),
    Total_Sales=('Sales','sum'), Total_Profit=('Profit','sum')
).reset_index().to_csv('dashboard_data/customer_segments.csv', index=False)

print('All dashboard files exported!')